# 🚀 Intelligent-AML: Master Physical Benchmark Suite
**IEEE Transactions on Information Forensics and Security (TIFS) / ACM KDD**

---

### ⚡ Accelerator: Kaggle TPU v5e-8 (224 vCPUs | 377 GB RAM)

This notebook executes the complete empirical benchmark pipeline:
1. **Phase 1**: 13 Models × 13 Datasets comparative benchmark
2. **Phase 2**: 24 Master Empirical Algorithmic Tests
3. **Phase 3**: Publication-grade LaTeX tables, vector figures, and statistical tests
4. **Phase 4**: 1-click ZIP download of all results

> ⚙️ In the right sidebar → **Session options** → **Accelerator** → select **TPU v5e-8** with **Internet ON**.

## Part 1: System Diagnostics & Thread Uncapping

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import psutil
cpus = psutil.cpu_count(logical=True) or 4
ram_gb = psutil.virtual_memory().total / (1024**3)
for env_var in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'POLARS_MAX_THREADS']:
    os.environ[env_var] = str(cpus)

import torch
torch.set_num_threads(cpus)

print('=' * 85)
print(' 🔍 SYSTEM HARDWARE PROFILE')
print('=' * 85)
print(f'• Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}')
print(f'• CPU Processors: {cpus} Logical Cores (High-Throughput Parallel SIMD Execution)')
print(f'• System Memory:  {ram_gb:.1f} GB RAM')
if torch.cuda.is_available():
    print(f'• Accelerator:    🚀 CUDA GPU ({torch.cuda.get_device_name(0)} x {torch.cuda.device_count()})')
else:
    print(f'• Accelerator:    ⚡ Kaggle TPU Cloud Host ({cpus} vCPUs & {ram_gb:.0f} GB RAM Active)')
print('=' * 85)

## Part 2: Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'polars', 'duckdb', 'catboost', 'lightgbm', 'xgboost', 'psutil',
    'scikit-learn', 'scipy', 'matplotlib', 'tabulate', 'torch_geometric', 'imbalanced-learn'
], check=True)
print('✓ All dependencies installed.')

## Part 3: Clone Repository & Mount Datasets
All bug fixes (SMOTE, AMP, thread caps, train_htgnn signature) are committed directly into the codebase — no fragile runtime patches needed.

In [ ]:
import os, sys, shutil
from pathlib import Path

repo = Path('/kaggle/working/Intelligent-AML').resolve()

# Clone or update
if not (repo / 'scripts' / 'run_automated_paper_benchmark.py').exists():
    !git clone https://github.com/NazmulHasanNihal/Intelligent-AML.git /kaggle/working/Intelligent-AML
else:
    !git -C /kaggle/working/Intelligent-AML fetch origin main
    !git -C /kaggle/working/Intelligent-AML reset --hard origin/main

# Suppress ALL warnings in subprocess children too
import warnings; warnings.filterwarnings('ignore')

# Safety: uncap threads in runner if somehow still capped
import psutil
for script in ['scripts/master_physical_benchmark_runner.py']:
    p = repo / script
    if p.exists():
        t = p.read_text(encoding='utf-8')
        changed = False
        for k in ['OMP_NUM_THREADS', 'POLARS_MAX_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
            old = f'os.environ["{k}"] = "2"'
            new = f'os.environ["{k}"] = str(max(2, __import__("psutil").cpu_count(logical=True)))'
            if old in t:
                t = t.replace(old, new); changed = True
        if changed:
            p.write_text(t, encoding='utf-8')
            print(f'  ✓ Uncapped threads in {script}')

# Safety: uncap base_models n_jobs
bm = repo / 'comparing_models' / 'base_models.py'
if bm.exists():
    t = bm.read_text(encoding='utf-8')
    if 'n_jobs=2' in t:
        t = t.replace('n_jobs=2', 'n_jobs=-1').replace('thread_count=2', 'thread_count=-1')
        bm.write_text(t, encoding='utf-8')
        print('  ✓ Uncapped base_models.py n_jobs')

# Set paths
os.chdir(str(repo))
for p in [str(repo), str(repo / 'scripts')]:
    if p not in sys.path: sys.path.insert(0, p)

# Mount datasets from /kaggle/input
graph_dir = repo / 'data' / 'outputs' / 'graph_data'
graph_dir.mkdir(parents=True, exist_ok=True)
cache_dir = repo / 'data' / 'cache'
cache_dir.mkdir(parents=True, exist_ok=True)

if Path('/kaggle/input').exists():
    for p in Path('/kaggle/input').rglob('graph_data'):
        if p.is_dir():
            mounted = 0
            for ds in sorted(p.iterdir()):
                if ds.is_dir():
                    dst = graph_dir / ds.name
                    if not dst.exists():
                        try: os.symlink(ds, dst)
                        except: shutil.copytree(ds, dst)
                    mounted += 1
            if mounted: print(f'✓ Mounted {mounted} datasets from {p}')
            break

    for c in Path('/kaggle/input').rglob('cache'):
        if c.is_dir():
            for f in c.glob('*.pt'):
                dst = cache_dir / f.name
                if not dst.exists():
                    try: os.symlink(f, dst)
                    except: shutil.copy2(f, dst)
            print(f'✓ Linked caches from {c}')
            break

print(f'\n✓ Working directory: {os.getcwd()}')

## Part 4: Registered Model Portfolio

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from scripts.run_automated_paper_benchmark import ALL_MODELS_REGISTRY
import pandas as pd
display(pd.DataFrame([{'#': i+1, 'Model': m['name'], 'Category': m['category'], 'Reference': m['paper_ref']} for i, m in enumerate(ALL_MODELS_REGISTRY)]))

## Part 5: Clean-Slate Execution Setup

In [ ]:
# Set to True to retrain ALL models from scratch; False to resume from checkpoints
CLEAN_SLATE = False

if CLEAN_SLATE:
    for d in ['results/benchmarks', 'results/metrics']:
        p = Path(d)
        if p.exists():
            for child in p.iterdir():
                if child.is_dir(): shutil.rmtree(child)
                else: child.unlink()
    print('🧹 Wiped previous results. Full clean-slate run initialized.')
else:
    !python scripts/master_physical_benchmark_runner.py --status

## Part 6: Phase 1 — Physical Comparative Benchmark (13 × 13)
Runs all 13 models across all 13 datasets with full TPU core utilization and clean output filtering:

In [ ]:
import subprocess, sys, re, psutil, os

ram = psutil.virtual_memory().total / (1024**3)
safe_ram = min(320.0, ram * 0.85)

# Force unbuffered stdout and suppress warnings
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONWARNINGS'] = 'ignore'
env['PYTHONDONTWRITEBYTECODE'] = '1'

print('=' * 85)
print(f' 🚀 PHASE 1: 13 DATASETS × 13 MODELS | {psutil.cpu_count(logical=True)} vCPUs | RAM Limit: {safe_ram:.0f} GB')
print('=' * 85)

cmd = [sys.executable, '-u', '-W', 'ignore', 'scripts/master_physical_benchmark_runner.py',
       '--epochs', '10', '--max-ram-gb', f'{safe_ram:.1f}', '--skip-phase2']
if CLEAN_SLATE: cmd.append('--force-rerun')

skip = [re.compile(p) for p in [
    r'FutureWarning', r'UserWarning', r'DeprecationWarning', r'RuntimeWarning',
    r'feature names', r'LGBMClassifier was fitted',
    r'torch\.cuda\.amp', r'torch\.amp', r'/usr/local/lib', r'/kaggle/working.*Warning',
    r'^\s+warnings\.warn', r'is deprecated'
]]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, universal_newlines=True, env=env)
try:
    while True:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            continue
        if any(p.search(line) for p in skip): continue
        s = line.strip()
        if not s: continue
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\n[Benchmark Interrupted by User]')

proc.wait()
print(f'\n{"✓ PHASE 1 COMPLETED!" if proc.returncode == 0 else f"⚠️ Exit code: {proc.returncode}"}')

## Part 7: Phase 2 — 24 Master Empirical Algorithmic Tests

In [ ]:
import warnings, importlib
warnings.filterwarnings('ignore')

print('=' * 85)
print(' 🔬 PHASE 2: 24 MASTER EMPIRICAL EVALUATION SUITE')
print('=' * 85)

try:
    import scripts.run_24_master_empirical_tests as r24_mod
    importlib.reload(r24_mod)
except Exception:
    pass
from scripts.run_24_master_empirical_tests import Master24EmpiricalSuite

try:
    suite = Master24EmpiricalSuite(force_rerun=CLEAN_SLATE)
except TypeError:
    suite = Master24EmpiricalSuite()

suite.run_all_with_resumption()
suite.save_reports()
print('\n✓ ALL 24 EMPIRICAL TESTS COMPLETED!')

## Part 8: Phase 3 — LaTeX Tables, Scorecards & Statistical Tests

In [ ]:
import pandas as pd, numpy as np, warnings
from pathlib import Path
from IPython.display import display
warnings.filterwarnings('ignore')

# Generate LaTeX tables
try:
    from scripts.generate_paper_tables import generate_latex_tables
    generate_latex_tables()
    print('✓ LaTeX tables generated.')
except Exception as e:
    print(f'Note: LaTeX tables: {e}')

csv = Path('results/metrics/master_detailed_benchmark_results.csv')
if csv.exists():
    df = pd.read_csv(csv)

    # Table 1: F1 Pivot
    print('\n' + '=' * 90)
    print(' 📊 TABLE 1: MACRO F1-SCORE (%) — ALL DATASETS × ALL MODELS')
    print('=' * 90)
    piv = df.pivot_table(index='dataset', columns='model_slug', values='f1_score', aggfunc='last') * 100
    display(piv.round(2).fillna('-'))

    # Table 2: Detailed Scorecard
    print('\n' + '=' * 90)
    print(' 📈 TABLE 2: DETAILED PERFORMANCE SCORECARD')
    print('=' * 90)
    cols = [c for c in ['dataset','model','f1_score','recall','precision','pr_auc','inference_latency_ms','throughput_samples_per_sec'] if c in df.columns]
    det = df[cols].copy()
    for c in ['f1_score','recall','precision','pr_auc']:
        if c in det.columns: det[c] = (det[c]*100).round(2)
    display(det.head(30))

    # Table 3: Wilcoxon
    print('\n' + '=' * 90)
    print(' 📐 TABLE 3: WILCOXON SIGNED-RANK (C-STGB vs BASELINES)')
    print('=' * 90)
    try:
        from scipy.stats import wilcoxon
        slug = 'proposed_c_stgb'
        if slug in piv.columns:
            cs = piv[slug].dropna()
            rows = []
            for bl in piv.columns:
                if bl == slug: continue
                bs = piv.loc[cs.index, bl].dropna()
                ci = cs.index.intersection(bs.index)
                if len(ci) >= 3:
                    d = cs.loc[ci] - bs.loc[ci]
                    if not (d == 0).all():
                        _, p = wilcoxon(cs.loc[ci], bs.loc[ci], alternative='greater')
                        rows.append({'Baseline': bl, 'N': len(ci),
                            'C-STGB F1': f'{cs.loc[ci].mean():.2f}%', 'Baseline F1': f'{bs.loc[ci].mean():.2f}%',
                            'Gain': f'+{(cs.loc[ci].mean()-bs.loc[ci].mean()):.2f}%',
                            'p-value': f'{p:.4e}', 'Significant': '***' if p<0.01 else ('*' if p<0.05 else 'No')})
            if rows: display(pd.DataFrame(rows))
    except Exception as e:
        print(f'Note: {e}')

    # Table 4: LaTeX code
    tex = Path('papers/IEEE_Research_Paper/tables/tab2_baseline_scorecard.tex')
    if tex.exists():
        print('\n' + '=' * 90)
        print(f' 📄 IEEE TABLE 2 LATEX ({tex.name})')
        print('=' * 90)
        print(tex.read_text(encoding='utf-8')[:2000])
else:
    print('⚠️ No benchmark results CSV found. Run Phase 1 first.')

## Part 9: Publication Figures (300 DPI)

In [ ]:
import subprocess, sys, warnings
from pathlib import Path
from IPython.display import Image, display
warnings.filterwarnings('ignore')

fig_script = Path('scripts/generate_all_publication_figures.py')
if fig_script.exists():
    subprocess.run([sys.executable, '-W', 'ignore', str(fig_script)], check=False)

fig_dir = Path('papers/IEEE_Research_Paper/figures')
figs = [
    ('PR-ROC Curves', 'fig1_pr_roc_curves.png'),
    ('Latency-Throughput Pareto', 'fig5_latency_pareto_frontier.png'),
    ('Multi-Dataset Radar', 'fig7_multi_dataset_radar.png'),
    ('System Architecture', 'fig6_system_architecture.png'),
    ('Adversarial Robustness', 'fig9_adversarial_camouflage_robustness.png'),
    ('All Datasets PR Curves', 'fig_all_datasets_pr_curves.png'),
]
for title, fn in figs:
    fp = fig_dir / fn
    if fp.exists():
        print(f'\n{title}:')
        display(Image(filename=str(fp), width=720))

# Also check results/figures
for fp in Path('results/figures').glob('*.png'):
    print(f'\n{fp.stem}:')
    display(Image(filename=str(fp), width=720))

## Part 10: Package All Results — 1-Click ZIP Download

In [ ]:
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
out = out_dir / 'intelligent_aml_full_results.zip'
if out.exists(): out.unlink()

print('Packaging all results...')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['results/benchmarks', 'results/metrics', 'results/figures',
                   'papers/IEEE_Research_Paper/tables', 'papers/IEEE_Research_Paper/figures']:
        for f in Path(folder).rglob('*'):
            if f.is_file() and f.suffix.lower() in ['.json', '.csv', '.md', '.tex', '.pdf', '.png', '.svg']:
                zf.write(f, f.relative_to(Path.cwd()))

    for doc in ['docs/Live_Physical_Benchmark_Progress.md', 'docs/Paper_Empirical_Scorecard.md',
                'docs/master_24_empirical_evaluations_report.md']:
        if Path(doc).exists(): zf.write(Path(doc), doc)

mb = out.stat().st_size / (1024*1024)
print(f'\n✓ Packaged: {out} ({mb:.1f} MB)')
display(FileLink(str(out)))